# Connectivity-prediction pretraining (topo-contrast pilot)

Runs `src/train_connectivity_pretrain.py` on the 38-case ImageCAS pilot subset — trains a siamese image encoder + relational head to predict topological connectivity between crop pairs, and compares it against a Euclidean-distance-only baseline (pilot-gate step 3, see `docs/PROJECT.md`).

**Before running:** upload `pilot_data.zip` to your Google Drive (e.g. into a `topo-contrast/` folder at your Drive root) — code comes from GitHub directly, no zip needed for that anymore. GPU is optional — the model is tiny (8³ patches) and this was never actually compute-bound; it's here mainly to escape local disk I/O issues, not for GPU speed. If you do want a GPU, select **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust this if you put the data zip somewhere else in your Drive
DRIVE_DIR = '/content/drive/MyDrive/topo-contrast'

import os
assert os.path.exists(f'{DRIVE_DIR}/pilot_data.zip'), (
    f'Expected {DRIVE_DIR}/pilot_data.zip — upload it there first, '
    'or change DRIVE_DIR above to wherever you put it.'
)

In [ ]:
%cd /content
!rm -rf topo-contrast
!git clone https://github.com/akashyall34/topo-contrast.git
%cd topo-contrast
!unzip -oq "$DRIVE_DIR/pilot_data.zip"
# pilot_data.zip preserves the original data/processed/pilot/<case_id>/ layout
!ls data/processed/pilot | wc -l   # should print 38

In [ ]:
!pip install -q networkx scikit-image scipy scikit-learn pyyaml
# torch/numpy are already present in the Colab runtime

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Pilot-gate step 1: sampling feasibility (fast, CPU-only, no images loaded)

In [ ]:
%cd /content/topo-contrast
!python scripts/check_sampling_feasibility.py --data-dir data/processed/pilot --config configs/pilot.yaml

## Pilot-gate step 2: shortcut-learning falsification baseline

In [ ]:
!python scripts/check_shortcut_baseline.py --data-dir data/processed/pilot --config configs/pilot.yaml --seed 0

## Pilot-gate step 3: train the siamese encoder + relational head, compare against the distance-only floor

`configs/pilot.yaml`'s `output_dir` points at a local Colab path by default here (not Drive) so checkpoint writes every epoch don't hit any sync layer — same lesson as the OneDrive issue this notebook exists to route around. Copy the checkpoint to Drive afterward if you want to keep it (last cell does this).

In [ ]:
import yaml
with open('configs/pilot.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['output_dir'] = '/content/outputs/pilot'
with open('configs/pilot.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print('output_dir set to', cfg['output_dir'])

In [ ]:
!PYTHONUNBUFFERED=1 python -m src.train_connectivity_pretrain --config configs/pilot.yaml

## Save the trained checkpoint back to Drive

In [ ]:
!mkdir -p "$DRIVE_DIR/outputs"
!cp /content/outputs/pilot/latest.pth "$DRIVE_DIR/outputs/latest.pth"
print('saved to', f'{DRIVE_DIR}/outputs/latest.pth')